In [1]:
# from utility.data import dataset_split
import pandas as pd
import torch
import numpy as np

/home/hyang/anaconda3/envs/clip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch_geometric

In [4]:
edges = pd.read_csv('data/twitch-gamer_edges.csv')
nodes = pd.read_csv('data/twitch-gamer_feat.csv')

In [39]:
from torch_geometric.datasets import HeterophilousGraphDataset
from torch_geometric import transforms as T
from torch_geometric.loader import ClusterData, ClusterLoader
from torch_geometric.utils import to_undirected, dense_to_sparse, to_dense_adj,add_self_loops
import torch
dataset = HeterophilousGraphDataset(root = "data/Questions", name = "Questions", transform=T.NormalizeFeatures())
data = dataset[0]
data.edge_index = to_undirected(data.edge_index)
device = torch.device("cuda:{}".format(0))

In [40]:
data = data.to(device)

In [41]:
a = data.x @ data.x.t()

In [42]:
adj_idx = to_dense_adj(data.edge_index)[0]

In [43]:
a = a * adj_idx

In [50]:
a[0].sum()

tensor(0.0033, device='cuda:0')

In [44]:
dense_to_sparse(a)

RuntimeError: nonzero is not supported for tensors with more than INT_MAX elements,   file a support request

In [56]:
cluster_data = ClusterData(data, num_parts=10)

Computing METIS partitioning...
Done!


In [57]:
train_loader = ClusterLoader(cluster_data, batch_size=1, shuffle=True,
                             num_workers=8)

In [58]:
i = 0
for batch in train_loader:
    i = i + 1
print(i)

10


In [53]:
from torch import is_tensor

In [54]:
not is_tensor(1)

True

In [55]:
batch.x.shape

torch.Size([244, 300])